# Analysis of the results of evaluating different embedding and retrieval models

In [1]:
import json
import pandas as pd

In [2]:
KG_DIR='/home/jmlon/Downloads/Papers/AI-KG/AI/KG'
runs = json.load(open(KG_DIR+"/reports/experiments-questions-kg-v1.json"))["runs"]
df = pd.json_normalize(runs)          # one row per run: settings.*, aggregate.*
per_question = pd.json_normalize(runs, "results", ["run_id", ["settings", "chunk_size"]])

In [3]:
df.head()

,run_id,batch_id,recorded_at,command,baseline,results,question_set.version_id,question_set.fingerprint,question_set.path,question_set.questions,...,aggregate.query_cost.p25,aggregate.query_cost.median,aggregate.query_cost.p75,aggregate.query_cost_mean,aggregate.query_cost_total,aggregate.priced_questions,aggregate.judge_cost_total,aggregate.errors,aggregate.reranked_questions,aggregate.total_tokens
0,ef55a0bc981e,d2938fb3ed2f,2026-10-31T14:10:10-05:00,eval,False,"[{'id': 'ji-2022-1', 'question': 'What data-qu...",kg-v1,b61a8db0f0e08fdb,/home/jmlon/Downloads/Papers/AI-KG/AI/KG/quest...,36,...,0.000183,0.000218,0.000241,0.000217,0.007797,36,0.017063,0,0,84333
1,a79607254703,7f813606c3a0,2026-10-31T14:10:38-05:00,eval,False,"[{'id': 'ji-2022-1', 'question': 'What data-qu...",kg-v1,b61a8db0f0e08fdb,/home/jmlon/Downloads/Papers/AI-KG/AI/KG/quest...,36,...,0.000191,0.000220,0.000272,0.000234,0.008438,36,0.017009,0,0,100739
2,8657b0656dcb,e61b5be2eee1,2026-10-31T14:11:06-05:00,eval,False,"[{'id': 'ji-2022-1', 'question': 'What data-qu...",kg-v1,b61a8db0f0e08fdb,/home/jmlon/Downloads/Papers/AI-KG/AI/KG/quest...,36,...,0.000272,0.000305,0.000371,0.000327,0.011786,36,0.016963,0,0,125806
3,9d4fc6c3c02e,d4d4a5232f0d,2026-10-31T14:11:34-05:00,eval,False,"[{'id': 'ji-2022-1', 'question': 'What data-qu...",kg-v1,b61a8db0f0e08fdb,/home/jmlon/Downloads/Papers/AI-KG/AI/KG/quest...,36,...,0.000540,0.000640,0.000694,0.000636,0.022888,36,0.016988,0,0,211381
4,4c0dfc17f869,c0d058144de7,2026-10-31T14:12:04-05:00,eval,False,"[{'id': 'ji-2022-1', 'question': 'What data-qu...",kg-v1,b61a8db0f0e08fdb,/home/jmlon/Downloads/Papers/AI-KG/AI/KG/quest...,36,...,0.000248,0.000286,0.000316,0.000282,0.010158,36,0.016697,0,36,84535


Comparison of scores and cost

In [4]:
df.groupby(["settings.llm_model", "settings.reranker"], dropna=False)[
    ["aggregate.mean_score", "aggregate.mean_coverage", "aggregate.query_cost.median"]
].mean()

aggregate.mean_score  \
settings.llm_model settings.reranker                              
openai/gpt-4o-mini voyageai/rerank-3-lite              4.175926   
                   NaN                                 3.814236   
openai/gpt-6-luna  voyageai/rerank-3-lite              4.000000   
                   NaN                                 3.750000   

                                           aggregate.mean_coverage  \
settings.llm_model settings.reranker                                 
openai/gpt-4o-mini voyageai/rerank-3-lite                 0.768750   
                   NaN                                    0.706843   
openai/gpt-6-luna  voyageai/rerank-3-lite                 0.738889   
                   NaN                                    0.687616   

                                           aggregate.query_cost.median  
settings.llm_model settings.reranker                                    
openai/gpt-4o-mini voyageai/rerank-3-lite                     0.000373  
                   NaN                                        0.000331  
openai/gpt-6-luna  voyageai/rerank-3-lite                     0.000275  
                   NaN                                        0.000171

## Last experiment: answering model and reranker (question set kg-v3)

Graded with the retrieved passages in view (`judge_protocol: reference+passages`) and priced at list price, so these runs are not comparable with kg-v1 or kg-v2. Best configuration first; at equal score, the cheaper one.

In [5]:
runs_v3 = json.load(open(KG_DIR + "/reports/experiments-questions-kg-v3.json"))["runs"]
v3 = pd.json_normalize(runs_v3)
with_evidence = sum(q.get("passage_hit") is not None for q in runs_v3[0]["results"])
# The `eval` runs only; the compare-embeddings batch has its own table below.
evals = v3[v3["command"] == "eval"].reset_index(drop=True)

summary = pd.DataFrame({
    "model": evals["settings.llm_model"],
    "embedding": evals["settings.embedding_model"],
    "reranker": evals["settings.reranker"].fillna("—"),
    "chunk": evals["settings.chunk_size"].astype(str) + "/" + evals["settings.chunk_overlap"].astype(str),
    "top_k": evals["settings.top_k"],
    "retrieval": evals["settings.retrieval"],
    "score": evals["aggregate.mean_score"].round(2),
    "passage_hit": (evals["aggregate.passage_hit_rate"] * with_evidence).round().astype(int).astype(str)
    + f"/{with_evidence}",
    "coverage": (evals["aggregate.mean_coverage"] * 100).round().astype(int).astype(str) + "%",
    "cited": (evals["aggregate.citation_rate"] * 100).round().astype(int).astype(str) + "%",
    "usd_per_query_list": evals["aggregate.query_list_cost.median"].round(6),
    "s_per_question": evals["aggregate.question_time.median_s"].round(1),
})
# Best score first; at equal score, the cheaper configuration.
summary = summary.sort_values(["score", "usd_per_query_list"], ascending=[False, True]).reset_index(drop=True)
display(summary)

best = summary.iloc[0]
print(
    f"Best configuration: {best.score}/5, passage hit {best.passage_hit}, coverage {best.coverage}, "
    f"${best.usd_per_query_list * 1000:.2f} per 1,000 queries at list price\n"
)
print(f"""global:
    defaults:
        llm_model: {best.model}
        embedding_model: {best.embedding}
index:
    chunk_size: {best.chunk.split('/')[0]}
    chunk_overlap: {best.chunk.split('/')[1]}
    top_k: {best.top_k}
    retrieval: {best.retrieval}
    reranker: {best.reranker if best.reranker != '—' else 'null'}""")

,model,embedding,reranker,chunk,top_k,retrieval,score,passage_hit,coverage,cited,usd_per_query_list,s_per_question
0,openai/gpt-6-luna,openai/text-embedding-3-small,voyageai/rerank-3-lite,1000/200,7,hybrid,4.83,33/33,97%,100%,0.000331,3.7
1,openai/gpt-6-luna,qwen/qwen3-embedding-8b,voyageai/rerank-3-lite,1000/200,7,hybrid,4.64,32/33,92%,100%,0.000317,4.0
2,openai/gpt-6-luna,qwen/qwen3-embedding-8b,—,1000/200,7,hybrid,4.61,28/33,93%,97%,0.000208,3.6
3,openai/gpt-4o-mini,openai/text-embedding-3-small,voyageai/rerank-3-lite,1000/200,7,hybrid,4.53,33/33,91%,94%,0.000408,3.5
4,openai/gpt-6-luna,openai/text-embedding-3-small,—,1000/200,7,hybrid,4.00,27/33,79%,97%,0.000210,3.5
5,openai/gpt-4o-mini,openai/text-embedding-3-small,—,1000/200,7,hybrid,3.81,27/33,72%,89%,0.000286,3.0


Best configuration: 4.83/5, passage hit 33/33, coverage 97%, $0.33 per 1,000 queries at list price

global:
    defaults:
        llm_model: openai/gpt-6-luna
        embedding_model: openai/text-embedding-3-small
index:
    chunk_size: 1000
    chunk_overlap: 200
    top_k: 7
    retrieval: hybrid
    reranker: voyageai/rerank-3-lite


## Embedding models (question set kg-v3, no reranker)

The latest `compare-embeddings` batch: every model with the same answering model, chunking, `top_k` and hybrid retrieval, and no reranker, so the embedding model alone decides which passages reach the answer. Ranked by passage hit, which is identical on every run, then by score, which moves by up to 0.25 between identical runs. `index_usd` is the cost of embedding the whole collection once, from the catalogue price; the cost per query hardly depends on the embedding model.

In [6]:
from pathlib import Path

from zotero_rag.config import load_config
from zotero_rag.embedding_comparison import stored_tokens
from zotero_rag_core.stores.chroma import ChromaStore

# The latest compare-embeddings batch: every model at the same chunking, retrieval and LLM.
comparisons = v3[v3["command"] == "compare-embeddings"]
emb = comparisons[comparisons["batch_id"] == comparisons["batch_id"].iloc[-1]]

# Building a store embeds every chunk once; all stores of one chunking hold the same text.
config = load_config(Path(KG_DIR))
corpus_tokens = stored_tokens(ChromaStore(Path(KG_DIR) / "chroma"))


def index_cost(model: str) -> float | None:
    spec = config.embedding_model_spec(model)
    return spec.cost(corpus_tokens) if spec is not None else None


passages = (emb["aggregate.passage_hit_rate"] * with_evidence).round().astype(int)
embedding_table = pd.DataFrame({
    "embedding_model": emb["settings.embedding_model"] + emb["baseline"].map({True: " (baseline)", False: ""}),
    "dim": emb["settings.embedding_dim"],
    "passage_hit": passages.astype(str) + f"/{with_evidence}",
    "score": emb["aggregate.mean_score"].round(2),
    "coverage": (emb["aggregate.mean_coverage"] * 100).round().astype(int).astype(str) + "%",
    "cited": (emb["aggregate.citation_rate"] * 100).round().astype(int).astype(str) + "%",
    "paper_hit": (emb["aggregate.hit_rate"] * 100).round().astype(int).astype(str) + "%",
    "index_usd": emb["settings.embedding_model"].map(index_cost).round(4),
    "usd_per_query_list": emb["aggregate.query_list_cost.median"].round(6),
    "s_per_question": emb["aggregate.question_time.median_s"].round(1),
    "_passages": passages,
})
# Passage hit first: it is the same on every run, while scores move by up to 0.25.
embedding_table = (
    embedding_table.sort_values(["_passages", "score"], ascending=False)
    .drop(columns="_passages")
    .reset_index(drop=True)
)
shared = emb.iloc[0]
reranker = shared["settings.reranker"]
print(
    f"{shared['settings.llm_model']} answering · top_k {shared['settings.top_k']} · "
    f"{shared['settings.chunk_size']}/{shared['settings.chunk_overlap']} chunks · "
    f"{shared['settings.retrieval']} retrieval · reranker: {'none' if pd.isna(reranker) else reranker} · "
    f"{corpus_tokens:,} tokens to index"
)
display(embedding_table)

openai/gpt-6-luna answering · top_k 7 · 1000/200 chunks · hybrid retrieval · reranker: none · 208,833 tokens to index


,embedding_model,dim,passage_hit,score,coverage,cited,paper_hit,index_usd,usd_per_query_list,s_per_question
0,qwen/qwen3-embedding-8b,4096,28/33,4.67,93%,97%,97%,0.0021,0.000204,3.8
1,voyageai/voyage-4,1024,28/33,4.50,89%,100%,100%,0.0125,0.000213,3.3
2,perplexity/pplx-embed-v1-0.6b,1024,28/33,4.36,87%,100%,100%,0.0008,0.000220,3.3
3,baai/bge-m3,1024,27/33,4.31,85%,97%,100%,0.0021,0.000216,3.4
4,google/gemini-embedding-001,3072,27/33,4.28,86%,100%,100%,0.0313,0.000212,3.8
5,openai/text-embedding-3-small (baseline),1536,27/33,4.22,85%,97%,100%,0.0042,0.000211,3.4
6,nvidia/nemotron-3.5-embed-1b:free,2048,27/33,4.00,81%,100%,100%,0.0000,0.000213,3.0
7,amazon/titan-embed-text-v2,1024,24/33,4.03,79%,100%,100%,0.0042,0.000216,3.4


## Finalists: text-embedding-3-small vs Qwen3-Embedding-8B, with and without the reranker

All with gpt-6-luna answering, `top_k` 7, 1000/200 chunks and hybrid retrieval. Every run counts, `eval` and `compare-embeddings` alike, so a configuration run twice shows both scores — the spread between them is the run-to-run noise.

- Qwen3 on its own scores about as well as text-embedding-3-small with the reranker, at about two thirds of the cost per query and with one provider fewer.
- With Qwen3 the reranker still finds more answering passages (28 → 32 of 33), but the score was already near its ceiling and does not rise with them.
- Passage hit, which does not change between runs, still favours text-embedding-3-small with the reranker (33/33): the safer choice for a larger library or harder questions, at about $0.12 more per thousand queries.

In [7]:
# The two finalists, with and without the reranker. Every gpt-6-luna run on kg-v3 counts,
# eval and compare-embeddings alike, so a configuration run twice shows both scores.
finalists = ["openai/text-embedding-3-small", "qwen/qwen3-embedding-8b"]
runs_luna = v3[
    (v3["settings.llm_model"] == "openai/gpt-6-luna")
    & v3["settings.embedding_model"].isin(finalists)
    & (v3["settings.top_k"] == 7)
].assign(
    reranker=lambda d: d["settings.reranker"].fillna("—"),
    passages=lambda d: (d["aggregate.passage_hit_rate"] * with_evidence).round().astype(int),
)

finalist_table = (
    runs_luna.groupby(["settings.embedding_model", "reranker"])
    .agg(
        runs=("run_id", "size"),
        scores=("aggregate.mean_score", lambda s: " / ".join(f"{x:.2f}" for x in s)),
        score_mean=("aggregate.mean_score", "mean"),
        passages=("passages", "max"),
        coverage=("aggregate.mean_coverage", "mean"),
        cited=("aggregate.citation_rate", "mean"),
        usd_per_query_list=("aggregate.query_list_cost.median", "mean"),
        usd_per_1000_queries=("aggregate.query_list_cost.median", lambda s: s.mean() * 1000),
    )
    .reset_index()
    .rename(columns={"settings.embedding_model": "embedding_model"})
    .sort_values("score_mean", ascending=False)
    .reset_index(drop=True)
)
finalist_table["passages"] = finalist_table["passages"].astype(str) + f"/{with_evidence}"
finalist_table["score_mean"] = finalist_table["score_mean"].round(2)
finalist_table["coverage"] = (finalist_table["coverage"] * 100).round().astype(int).astype(str) + "%"
finalist_table["cited"] = (finalist_table["cited"] * 100).round().astype(int).astype(str) + "%"
finalist_table["usd_per_query_list"] = finalist_table["usd_per_query_list"].round(6)
finalist_table["usd_per_1000_queries"] = finalist_table["usd_per_1000_queries"].round(2)
display(finalist_table)

,embedding_model,reranker,runs,scores,score_mean,passages,coverage,cited,usd_per_query_list,usd_per_1000_queries
0,openai/text-embedding-3-small,voyageai/rerank-3-lite,1,4.83,4.83,33/33,97%,100%,0.000331,0.33
1,qwen/qwen3-embedding-8b,voyageai/rerank-3-lite,1,4.64,4.64,32/33,92%,100%,0.000317,0.32
2,qwen/qwen3-embedding-8b,—,2,4.67 / 4.61,4.64,28/33,93%,97%,0.000206,0.21
3,openai/text-embedding-3-small,—,2,4.00 / 4.22,4.11,27/33,82%,97%,0.000211,0.21


## Plots

All three question sets loaded into one table, one row per run. Scores from different question sets are not comparable, so the plots put each set in its own panel. Costs are in USD per 1,000 queries: at list price for kg-v3, and as charged for kg-v1 and kg-v2, where the provider's cache discount makes identical runs differ by up to 4×. Hover over any mark for its full configuration.

In [16]:
import json

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

KG_DIR = "/home/jmlon/Downloads/Papers/AI-KG/AI/KG"
QUESTION_SETS = ["kg-v1", "kg-v2", "kg-v3"]

# Reference palette (validated: the first three categorical slots clear every check all-pairs).
SURFACE, TEXT, TEXT_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
BLUE, ORANGE, AQUA, RED, NEUTRAL = "#2a78d6", "#eb6834", "#1baf7a", "#e34948", "#f0efec"
SEQUENTIAL = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
RERANK_COLOR = {False: BLUE, True: ORANGE}
MODEL_SYMBOL = {"gpt-4o-mini": "circle", "gpt-6-luna": "diamond"}


def short(model: str | None) -> str:
    return model.split("/")[-1] if model else "none"


def mean_of(results: list[dict], field: str) -> float | None:
    values = [q[field] for q in results if q.get(field) is not None]
    return sum(values) / len(values) if values else None


rows, question_rows = [], []
for version in QUESTION_SETS:
    raw = json.load(open(f"{KG_DIR}/reports/experiments-questions-{version}.json"))["runs"]
    with_quotes = max(sum(q.get("passage_hit") is not None for q in r["results"]) for r in raw)
    for r in raw:
        s, a, results = r["settings"], r["aggregate"], r["results"]
        # List price where the run recorded it (kg-v3); before that, only the actual charge.
        list_price = (a.get("query_list_cost") or {}).get("median")
        config = (
            f"{short(s['llm_model'])} · {short(s['embedding_model'])} · "
            f"{s['chunk_size']}/{s['chunk_overlap']} · k={s['top_k']} · "
            f"{short(s.get('reranker')) if s.get('reranker') else 'no reranker'}"
        )
        rows.append({
            "question_set": version,
            "run_id": r["run_id"],
            "config": config,
            "llm": short(s["llm_model"]),
            "embedding": short(s["embedding_model"]),
            "chunk": f"{s['chunk_size']}/{s['chunk_overlap']}",
            "chunk_size": s["chunk_size"],
            "top_k": s["top_k"],
            "reranked": bool(s.get("reranker")),
            "score": a["mean_score"],
            "passage_rate": a.get("passage_hit_rate"),
            "passages": round((a.get("passage_hit_rate") or 0) * with_quotes),
            "with_quotes": with_quotes,
            "coverage": a.get("mean_coverage"),
            "usd_per_1k": 1000 * (list_price if list_price is not None else (a.get("query_cost") or {}).get("median") or 0),
            "cost_basis": "list price" if list_price is not None else "charged",
            "s_per_question": (a.get("question_time") or {}).get("median_s"),
            # Mean per query, for the cost breakdown.
            "answer_per_1k": 1000 * (mean_of(results, "answer_list_cost" if list_price is not None else "answer_cost") or 0),
            "rerank_per_1k": 1000 * (mean_of(results, "rerank_cost") or 0),
            "embedding_per_1k": 1000 * (mean_of(results, "embedding_cost") or 0),
        })
        for q in results:
            question_rows.append({
                "question_set": version, "config": config, "question": q["id"],
                "score": q["score"], "grade": q["grade"], "passage_hit": q.get("passage_hit"),
            })

runs_all = pd.DataFrame(rows)
questions_all = pd.DataFrame(question_rows)


def styled(fig: go.Figure, height: int) -> go.Figure:
    fig.update_layout(
        template="plotly_white", height=height, paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
        font=dict(color=TEXT, size=12), margin=dict(l=60, r=30, t=70, b=50),
        legend=dict(orientation="h", y=-0.18, x=0, font=dict(color=TEXT_2)),
        hoverlabel=dict(bgcolor="white", font_color=TEXT),
    )
    fig.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=GRID, title_font_color=TEXT_2)
    fig.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=GRID, title_font_color=TEXT_2)
    fig.update_annotations(font_color=TEXT)
    return fig


runs_all.groupby("question_set").size()

question_set
kg-v1    23
kg-v2     4
kg-v3    14
dtype: int64

### Chunking × top_k

Only kg-v1 varied the chunking and `top_k`, so this uses kg-v1 (gpt-4o-mini, no reranker, the earlier grading). Repeat runs are averaged; `n` is how many. Passage hit is identical on every run, so it is the steadier of the two for comparing cells.

In [17]:
def chunk_topk_heatmaps() -> go.Figure:
    """kg-v1 only: the one set that varied chunking and top_k. gpt-4o-mini, no reranker."""
    grid = runs_all[
        (runs_all["question_set"] == "kg-v1") & (runs_all["llm"] == "gpt-4o-mini") & ~runs_all["reranked"]
    ]
    cells = grid.groupby(["chunk_size", "chunk", "top_k"]).agg(
        score=("score", "mean"), runs=("score", "size"), passages=("passages", "mean"), of=("with_quotes", "max")
    ).reset_index()
    chunks = cells.sort_values("chunk_size")["chunk"].unique().tolist()
    ks = sorted(cells["top_k"].unique())
    score = cells.pivot(index="chunk", columns="top_k", values="score").reindex(index=chunks, columns=ks)
    passages = cells.pivot(index="chunk", columns="top_k", values="passages").reindex(index=chunks, columns=ks)
    runs = cells.pivot(index="chunk", columns="top_k", values="runs").reindex(index=chunks, columns=ks)
    of = int(cells["of"].max())

    score_text = [[f"{v:.2f}<br>n={int(n)}" if pd.notna(v) else "" for v, n in zip(sr, nr)]
                  for sr, nr in zip(score.values, runs.values)]
    passage_text = [[f"{int(v)}/{of}" if pd.notna(v) else "" for v in pr] for pr in passages.values]
    scale = [[i / (len(SEQUENTIAL) - 1), c] for i, c in enumerate(SEQUENTIAL)]
    columns = [f"k={k}" for k in ks]

    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.16,
                        subplot_titles=("Mean score (out of 5)", f"Passage hit (of {of} questions)"))
    fig.add_trace(go.Heatmap(
        z=score.values, x=columns, y=chunks, text=score_text, texttemplate="%{text}",
        colorscale=scale, xgap=2, ygap=2, colorbar=dict(x=0.42, len=0.9, title=""),
        hovertemplate="chunk %{y} · %{x}<br>mean score %{z:.2f}<extra></extra>",
    ), row=1, col=1)
    fig.add_trace(go.Heatmap(
        z=passages.values, x=columns, y=chunks, text=passage_text, texttemplate="%{text}",
        colorscale=scale, xgap=2, ygap=2, colorbar=dict(x=1.0, len=0.9, title=""),
        hovertemplate=f"chunk %{{y}} · %{{x}}<br>passage hit %{{z:.0f}}/{of}<extra></extra>",
    ), row=1, col=2)
    fig.update_yaxes(title_text="chunk size / overlap (characters)", row=1, col=1)
    fig.update_xaxes(title_text="top_k", row=1, col=1)
    fig.update_xaxes(title_text="top_k", row=1, col=2)
    fig.update_layout(title=dict(text="Chunking × top_k · kg-v1 · gpt-4o-mini, hybrid retrieval, no reranker",
                                 font_size=15))
    styled(fig, 360)
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=False)
    return fig


chunk_topk_heatmaps().show()

### Score against cost and time

One dot per run. Colour shows the reranker, shape the answering model. On the cost plot, the line joins the runs that nothing beats on both score and cost — the configurations worth choosing between.

In [10]:
def pareto(frame: pd.DataFrame, x: str) -> pd.DataFrame:
    """The runs no other run beats on both a higher score and a lower `x`."""
    best, front = float("-inf"), []
    for _, row in frame.sort_values([x, "score"], ascending=[True, False]).iterrows():
        if row["score"] > best:
            front.append(row)
            best = row["score"]
    return pd.DataFrame(front)


def score_scatter(x: str, x_title: str, title: str, frontier: bool = False) -> go.Figure:
    """One panel per question set: their scores are not comparable with one another."""
    fig = make_subplots(rows=1, cols=len(QUESTION_SETS), subplot_titles=QUESTION_SETS, horizontal_spacing=0.07)
    shown = set()
    for col, version in enumerate(QUESTION_SETS, start=1):
        panel = runs_all[runs_all["question_set"] == version]
        if frontier:
            front = pareto(panel, x)
            fig.add_trace(go.Scatter(
                x=front[x], y=front["score"], mode="lines", line=dict(color=TEXT_2, width=1.5),
                name="best trade-offs (nothing is both better and cheaper)", legendgroup="front",
                showlegend="front" not in shown, hoverinfo="skip",
            ), row=1, col=col)
            shown.add("front")
        for (reranked, llm), group in panel.groupby(["reranked", "llm"]):
            key = (reranked, llm)
            fig.add_trace(go.Scatter(
                x=group[x], y=group["score"], mode="markers",
                marker=dict(color=RERANK_COLOR[reranked], symbol=MODEL_SYMBOL.get(llm, "square"), size=12,
                            line=dict(color=SURFACE, width=2)),
                name=f"{llm} · {'reranker' if reranked else 'no reranker'}", legendgroup=str(key),
                showlegend=key not in shown,
                customdata=group[["config", "passages", "with_quotes", "usd_per_1k", "cost_basis", "s_per_question"]],
                hovertemplate=(
                    "%{customdata[0]}<br>score %{y:.2f} · passages %{customdata[1]}/%{customdata[2]}"
                    "<br>$%{customdata[3]:.3f} per 1,000 queries (%{customdata[4]})"
                    "<br>%{customdata[5]:.1f} s per question<extra></extra>"
                ),
            ), row=1, col=col)
            shown.add(key)
        basis = panel["cost_basis"].iloc[0] if x == "usd_per_1k" else ""
        fig.update_xaxes(title_text=x_title + (f" ({basis})" if basis else ""), row=1, col=col)
    fig.update_yaxes(title_text="mean score (out of 5)", row=1, col=1)
    fig.update_layout(title=dict(text=title, font_size=15))
    return styled(fig, 430)

In [11]:
score_scatter(
    "usd_per_1k", "USD per 1,000 queries",
    "Score vs cost per query · one dot per run · the line joins the best trade-offs",
    frontier=True,
).show()

In [12]:
score_scatter(
    "s_per_question", "median seconds per question",
    "Score vs time per question · one dot per run",
).show()

### What a query costs

Mean cost per query of each configuration, split into the answer, reranking and embedding the question; repeat runs are averaged. Embedding the question is a few tokens, so its segment only shows for the dearest model (gemini-embedding-001). Grading is evaluation overhead and is not part of a query's cost.

In [13]:
def cost_breakdown() -> go.Figure:
    """Mean cost per query of each configuration, split into its parts; repeat runs averaged."""
    parts = [("answer_per_1k", "LLM answer", BLUE), ("rerank_per_1k", "reranking", ORANGE),
             ("embedding_per_1k", "question embedding", AQUA)]
    by_config = (
        runs_all.groupby(["question_set", "config", "cost_basis"])[[p for p, _, _ in parts]]
        .mean().reset_index()
    )
    by_config["total"] = by_config[[p for p, _, _ in parts]].sum(axis=1)
    counts = [len(by_config[by_config["question_set"] == v]) for v in QUESTION_SETS]
    fig = make_subplots(
        rows=len(QUESTION_SETS), cols=1, row_heights=counts, vertical_spacing=0.07,
        subplot_titles=[
            f"{v} · USD per 1,000 queries, {by_config[by_config['question_set'] == v]['cost_basis'].iloc[0]}"
            for v in QUESTION_SETS
        ],
    )
    for row, version in enumerate(QUESTION_SETS, start=1):
        panel = by_config[by_config["question_set"] == version].sort_values("total")
        for field, label, color in parts:
            fig.add_trace(go.Bar(
                y=panel["config"], x=panel[field], orientation="h", name=label, legendgroup=label,
                showlegend=row == 1, marker=dict(color=color, line=dict(color=SURFACE, width=2)),
                hovertemplate=f"%{{y}}<br>{label}: $%{{x:.4f}} per 1,000 queries<extra></extra>",
            ), row=row, col=1)
    fig.update_layout(barmode="stack", bargap=0.35,
                      title=dict(text="What a query costs · mean per query, repeat runs averaged", font_size=15))
    styled(fig, 230 + 26 * sum(counts))
    fig.update_layout(margin=dict(l=420), legend=dict(y=-0.06))
    fig.update_yaxes(showgrid=False, tickfont=dict(size=11, color=TEXT_2))
    return fig


cost_breakdown().show()

### Passage hit against score

Does retrieving the answering passage translate into a better answer? Dots far below the trend found the passage and still answered poorly — an answering problem, not a retrieval one.

In [14]:
def passage_vs_score() -> go.Figure:
    fig = make_subplots(rows=1, cols=len(QUESTION_SETS), subplot_titles=QUESTION_SETS, horizontal_spacing=0.07)
    shown = set()
    for col, version in enumerate(QUESTION_SETS, start=1):
        panel = runs_all[runs_all["question_set"] == version]
        for (reranked, llm), group in panel.groupby(["reranked", "llm"]):
            key = (reranked, llm)
            fig.add_trace(go.Scatter(
                x=100 * group["passage_rate"], y=group["score"], mode="markers",
                marker=dict(color=RERANK_COLOR[reranked], symbol=MODEL_SYMBOL.get(llm, "square"), size=12,
                            line=dict(color=SURFACE, width=2)),
                name=f"{llm} · {'reranker' if reranked else 'no reranker'}", legendgroup=str(key),
                showlegend=key not in shown, customdata=group[["config", "passages", "with_quotes"]],
                hovertemplate="%{customdata[0]}<br>passages %{customdata[1]}/%{customdata[2]} · score %{y:.2f}<extra></extra>",
            ), row=1, col=col)
            shown.add(key)
        fig.update_xaxes(title_text="passage hit (%)", row=1, col=col)
    fig.update_yaxes(title_text="mean score (out of 5)", row=1, col=1)
    fig.update_layout(title=dict(text="Does finding the passage give a better answer? · one dot per run", font_size=15))
    return styled(fig, 430)


passage_vs_score().show()

### Each question under each configuration

Repeat runs averaged. Questions that fail across most rows are hard or flawed; questions that fail only in some rows depend on the setup. Change the argument to see `kg-v1` or `kg-v2`.

In [15]:
def question_grid(version: str = "kg-v3") -> go.Figure:
    """Each question's score under each configuration (repeat runs averaged).

    Configurations run down the side, best on top; questions across, hardest on the left.
    Labels keep only the settings that differ between this set's configurations.
    """
    panel = questions_all[questions_all["question_set"] == version]
    table = panel.pivot_table(index="config", columns="question", values="score", aggfunc="mean")
    table = table.loc[table.mean(axis=1).sort_values(ascending=False).index]
    table = table[table.mean(axis=0).sort_values().index]
    parts = [c.split(" · ") for c in table.index]
    varying = [i for i in range(len(parts[0])) if len({p[i] for p in parts}) > 1]
    labels = [" · ".join(p[i] for i in varying) for p in parts]
    shared = " · ".join(parts[0][i] for i in range(len(parts[0])) if i not in varying)
    fig = go.Figure(go.Heatmap(
        z=table.values, x=table.columns, y=labels, zmin=0, zmax=5, xgap=2, ygap=2,
        colorscale=[[0, RED], [0.5, NEUTRAL], [1, BLUE]],
        colorbar=dict(title="score", tickvals=[0, 1, 2, 4, 5],
                      ticktext=["0 wrong / none", "1 poor", "2 partial", "4 good", "5 perfect"]),
        hovertemplate="%{y}<br>%{x}<br>score %{z:.1f}<extra></extra>",
    ))
    fig.update_layout(title=dict(
        text=f"Each question under each configuration · {version}<br>"
             f"<span style='font-size:12px;color:{TEXT_2}'>all rows: {shared} · best configuration on top, "
             f"hardest questions on the left</span>",
        font_size=15,
    ))
    styled(fig, 230 + 26 * table.shape[0])
    fig.update_layout(margin=dict(l=330, t=90, b=120))
    fig.update_xaxes(tickangle=-60, tickfont=dict(size=10, color=TEXT_2), showgrid=False)
    fig.update_yaxes(autorange="reversed", tickfont=dict(size=11, color=TEXT_2), showgrid=False)
    return fig


question_grid("kg-v3").show()

## Embedding norms: does the distance metric matter?

Cosine similarity, dot product and Euclidean distance rank neighbours identically when every vector has length 1: then cosine = a·b and ‖a−b‖² = 2 − 2·a·b. Chroma index folders use squared L2 and the PostgreSQL store cosine, so a metric comparison only means something for a model whose vectors are not unit length. This reads a sample of the stored chunk embeddings of every store in the index folder and reports their L2 norms. `metrics_agree` is true when every sampled vector is within 1e-3 of unit length. Where it is false, cosine and Euclidean still nearly agree, but the dot product favours longer vectors; a small `norm_cv` (spread of the norms) means even that effect is small.

In [ ]:
import chromadb
import numpy as np
from chromadb.config import Settings
from pathlib import Path

from zotero_rag.config import load_config
from zotero_rag_core.stores.chroma import COLLECTION_NAME

#: Chunks read per store. A few thousand settle the question; reading every vector is slow.
NORM_SAMPLE = 5000
#: |norm - 1| below this counts as unit length (float32 rounding is ~1e-7).
UNIT_TOLERANCE = 1e-3


def embedding_norms(store_dir: Path, sample: int = NORM_SAMPLE) -> np.ndarray:
    """L2 norms of up to `sample` stored chunk embeddings, read straight from Chroma."""
    client = chromadb.PersistentClient(path=str(store_dir), settings=Settings(anonymized_telemetry=False))
    collection = client.get_collection(COLLECTION_NAME)
    vectors = np.asarray(collection.get(limit=sample, include=["embeddings"])["embeddings"], dtype=np.float64)
    return np.linalg.norm(vectors, axis=1)


def norm_table(index_dir: Path) -> pd.DataFrame:
    """One row per store in the index folder: the baseline `chroma/` and every compare-embeddings variant."""
    baseline_model = load_config(index_dir).embedding_model
    stores = [(f"{baseline_model} (baseline)", "chroma", index_dir / "chroma")]
    stores += [
        (model_dir.name, chunking.name, chunking)  # model_dir is the model's folder slug
        for model_dir in sorted((index_dir / "embeddings").glob("*"))
        for chunking in sorted(model_dir.glob("*"))
        if (chunking / "chroma.sqlite3").exists()
    ]
    rows = []
    for model, chunking, store_dir in stores:
        norms = embedding_norms(store_dir)
        if norms.size == 0:
            continue
        unit = np.abs(norms - 1) < UNIT_TOLERANCE
        rows.append({
            "embedding_model": model,
            "chunking": chunking,
            "chunks_read": norms.size,
            "norm_min": norms.min(),
            "norm_mean": norms.mean(),
            "norm_max": norms.max(),
            # Spread relative to the mean: 0 means every vector has the same length, so the
            # dot product ranks like cosine even when that length is not 1.
            "norm_cv": norms.std() / norms.mean(),
            "unit_length": f"{unit.mean():.1%}",
            "metrics_agree": bool(unit.all()),
        })
    return pd.DataFrame(rows).round(6)


norms = norm_table(Path(KG_DIR))
display(norms)
differ = norms.loc[~norms["metrics_agree"], "embedding_model"].tolist()
print(
    "Every store holds unit-length vectors: cosine, dot product and Euclidean distance rank identically."
    if not differ
    else f"Not unit length, so dot product can rank differently from cosine: {', '.join(differ)}"
)